# 07. ONNX Model Export & Deployment Verification — EMIPredict AI

This notebook loads the Production-stage models (`best_classifier.pkl`, `best_regressor.pkl`) and `scaler.pkl`, wraps them inside an end-to-end `sklearn.pipeline.Pipeline([('scaler', scaler), ('model', model)])`, and exports them to open-standard ONNX format (`best_classifier.onnx`, `best_regressor.onnx`).

**Key Architecture (D4)**: Baking `StandardScaler` directly into the ONNX graph ensures the Node.js backend can feed the raw unscaled 48-feature vector into ONNX with zero runtime drift and zero manual client-side scaling.

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd

# Add scripts to path for canonical feature utils
scripts_path = '../scripts' if os.path.exists('../scripts') else 'scripts'
if scripts_path not in sys.path:
    sys.path.insert(0, os.path.abspath(scripts_path))

from feature_utils import build_feature_vector, ALL_FEATURE_ORDER

# Paths Verification
base_models = '../models' if os.path.exists('../models') else 'models'
class_path = os.path.join(base_models, 'classification', 'best_classifier.pkl')
reg_path = os.path.join(base_models, 'regression', 'best_regressor.pkl')
enc_path = os.path.join(base_models, 'preprocessing', 'encoders.pkl')
scaler_path = os.path.join(base_models, 'preprocessing', 'scaler.pkl')

class_onnx_path = os.path.join(base_models, 'classification', 'best_classifier.onnx')
reg_onnx_path = os.path.join(base_models, 'regression', 'best_regressor.onnx')

print(f"Total features in ALL_FEATURE_ORDER: {len(ALL_FEATURE_ORDER)}")
print(f"Scaler exists: {os.path.exists(scaler_path)}")

Total features in ALL_FEATURE_ORDER: 48
Scaler exists: True


In [2]:
# 1. Export ONNX models (unscaled) + scaler parameters as JSON (D4, revised)
import json
import onnx
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

NUM_COUNT = 24  # first 24 columns of ALL_FEATURE_ORDER are numeric/derived (scaled); remaining 24 are one-hot (untouched)
initial_types = [('float_input', FloatTensorType([None, len(ALL_FEATURE_ORDER)]))]

scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

if scaler is not None:
    scaler_params = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "applies_to_first_n_columns": NUM_COUNT
    }
    with open('../models/preprocessing/scaler_params.json', 'w') as f:
        json.dump(scaler_params, f, indent=2)
    print(f"\u2713 Exported scaler parameters ({NUM_COUNT} columns) to scaler_params.json")

if os.path.exists(class_path):
    clf_raw = joblib.load(class_path)
    print(f"Loaded Classifier: {type(clf_raw).__name__}")
    if 'XGB' in type(clf_raw).__name__:
        import onnxmltools
        onx_clf = onnxmltools.convert_xgboost(clf_raw, initial_types=initial_types)
    else:
        onx_clf = convert_sklearn(clf_raw, initial_types=initial_types, options={id(clf_raw): {'zipmap': False}})
    onnx.save_model(onx_clf, class_onnx_path)
    print(f"\u2713 Saved classifier ONNX to: {class_onnx_path}")

if os.path.exists(reg_path):
    reg_raw = joblib.load(reg_path)
    print(f"Loaded Regressor: {type(reg_raw).__name__}")
    if 'XGB' in type(reg_raw).__name__:
        import onnxmltools
        onx_reg = onnxmltools.convert_xgboost(reg_raw, initial_types=initial_types)
    else:
        onx_reg = convert_sklearn(reg_raw, initial_types=initial_types)
    onnx.save_model(onx_reg, reg_onnx_path)
    print(f"\u2713 Saved regressor ONNX to: {reg_onnx_path}")


✓ Exported scaler parameters (24 columns) to scaler_params.json
Loaded Classifier: XGBClassifier


✓ Saved classifier ONNX to: ../models/classification/best_classifier.onnx
Loaded Regressor: XGBRegressor


✓ Saved regressor ONNX to: ../models/regression/best_regressor.onnx


In [3]:
# 2. Parity Verification: scale explicitly (first 24 cols only) on both sides, then compare
if os.path.exists(class_onnx_path) and os.path.exists(reg_onnx_path):
    sample_payload = {
        'age': 32, 'gender': 'Male', 'marital_status': 'Married', 'education': 'Graduate',
        'monthly_salary': 65000, 'employment_type': 'Private', 'years_of_employment': 4.5,
        'company_type': 'MNC', 'house_type': 'Rented', 'monthly_rent': 12000, 'family_size': 3,
        'dependents': 1, 'school_fees': 2500, 'college_fees': 0, 'travel_expenses': 3500,
        'groceries_utilities': 8000, 'other_monthly_expenses': 3000, 'existing_loans': True,
        'current_emi_amount': 5000, 'credit_score': 750, 'bank_balance': 120000,
        'emergency_fund': 45000, 'emi_scenario': 'Personal Loan', 'requested_amount': 150000,
        'requested_tenure': 24
    }

    raw_vector, derived = build_feature_vector(sample_payload, encoder=None, scaler=None)
    X_raw = np.array([raw_vector], dtype=np.float32)

    clf_pkl = joblib.load(class_path)
    reg_pkl = joblib.load(reg_path)
    scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

    X_scaled = X_raw.copy()
    if scaler is not None:
        X_scaled[:, :NUM_COUNT] = scaler.transform(X_raw[:, :NUM_COUNT])

    pkl_class = clf_pkl.predict(X_scaled)[0]
    pkl_emi = float(reg_pkl.predict(X_scaled)[0])

    sess_clf = ort.InferenceSession(class_onnx_path)
    sess_reg = ort.InferenceSession(reg_onnx_path)
    input_name_clf = sess_clf.get_inputs()[0].name
    input_name_reg = sess_reg.get_inputs()[0].name

    onnx_class_out = sess_clf.run(None, {input_name_clf: X_scaled})[0][0]
    onnx_emi_out = float(sess_reg.run(None, {input_name_reg: X_scaled})[0][0])

    print("\n=== PARITY VERIFICATION (explicit scaling, both sides) ===")
    print(f"  Classification: Scaled PKL={pkl_class}, ONNX={onnx_class_out} -> {'PASS' if str(pkl_class) == str(onnx_class_out) else 'FAIL'}")
    print(f"  Regression Max EMI: Scaled PKL=\u20b9{pkl_emi:,.2f}, ONNX=\u20b9{onnx_emi_out:,.2f} -> {'PASS' if abs(pkl_emi - onnx_emi_out) < 1.0 else 'FAIL'}")

    assert str(pkl_class) == str(onnx_class_out), "Classification ONNX output divergence!"
    assert abs(pkl_emi - onnx_emi_out) < 1.0, "Regression ONNX output divergence!"
    print("\nONNX EXPORT VERIFIED: EXPLICIT SCALING PARITY CONFIRMED")
else:
    print("\nONNX models will be generated when notebook 07 is executed in Colab.")



=== PARITY VERIFICATION (explicit scaling, both sides) ===
  Classification: Scaled PKL=0, ONNX=0 -> PASS
  Regression Max EMI: Scaled PKL=₹15,389.64, ONNX=₹15,389.63 -> PASS

ONNX EXPORT VERIFIED: EXPLICIT SCALING PARITY CONFIRMED


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/tmp/ipykernel_55325/1706894836.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  onnx_emi_out = float(sess_reg.run(None, {input_name_reg: X_scaled})[0][0])
